# Lekcja 12 - Redukcja Historii Czatów za pomocą Notatnika Agenta

Ten notatnik pokazuje, jak zarządzać kontekstem w długich rozmowach przy użyciu Microsoft Agent Framework. W miarę rozwoju rozmów liczba tokenów rośnie — ostatecznie przekraczając okno kontekstowe modelu. Rozwiązujemy to za pomocą **wzoru podsumowywania kontekstu** oraz **notatnika agenta** dla trwałej pamięci.

## Czego się nauczysz:
1. **Dlaczego zarządzanie kontekstem ma znaczenie**: Zrozumienie limitów tokenów i okien kontekstowych
2. **Agenci świadomi kontekstu**: Budowanie agentów zarządzających własnym kontekstem rozmowy
3. **Wzór podsumowywania kontekstu**: Używanie narzędzi do kondensacji historii rozmów
4. **Notatnik agenta**: Trwała pamięć, która przetrwa redukcję kontekstu

## Wymagania wstępne:
- Konfiguracja Azure OpenAI z ustawionymi zmiennymi środowiskowymi
- Zrozumienie podstawowych pojęć agentów z poprzednich lekcji


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Dlaczego zarządzanie kontekstem ma znaczenie

Każdy LLM ma ograniczone **okno kontekstu** — maksymalną liczbę tokenów, które może przetworzyć w jednym żądaniu. W miarę jak przebiega rozmowa wieloetapowa:

- **Liczba tokenów rośnie liniowo** z każdą wiadomością użytkownika i odpowiedzią asystenta.
- **Tokeny prompta dominują koszty**, ponieważ cała historia jest wysyłana na nowo przy każdym kroku.
- Ostatecznie rozmowa **przekracza okno kontekstu**, a model albo przycina, albo zgłasza błąd.

### Strategie zarządzania kontekstem

| Strategia | Jak działa | Kompromis |
|---|---|---|
| **Przycinanie** | Usuwa najstarsze wiadomości | Traci wczesny kontekst |
| **Podsumowanie** | Kondensuje starsze wiadomości do podsumowania | Część szczegółów zostaje utracona, ale kluczowe punkty są zachowane |
| **Scratchpad / Pamięć zewnętrzna** | Przechowuje kluczowe fakty poza rozmową | Wymaga wywołań narzędzi, ale przetrwa każde skrócenie |

W tym notebooku łączymy **podsumowanie** z **narzędziem scratchpad**, aby agent mógł utrzymać ciągłość nawet, gdy historia rozmowy jest kondensowana.


## Tworzenie agenta świadomego kontekstu


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Symulacja Długiej Konwersacji

Przejdźmy przez rozmowę wieloetapową, aby zobaczyć, jak kontekst się kumuluje. Agent powinien zapamiętać kluczowe szczegóły (preferencje, budżet, daty podróży) podczas kolejnych etapów i wykazać ciągłość.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Zauważ, jak agent zachowuje kontekst z wcześniejszych tur — wie o Japonii, sushi, świątyniach, fotografii, budżecie 3000 dolarów, podróży solo i terminie w kwietniu. W krótkiej rozmowie działa to dobrze, ale w miarę rozwoju rozmowy pełna historia staje się kosztowna do ponownego przesłania.

Kontynuujmy rozmowę z większą liczbą tur, aby zobaczyć gromadzenie kontekstu:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Wzorzec podsumowania kontekstu

W miarę rozwoju rozmowy możemy użyć **narzędzia do podsumowywania**, aby skondensować zebrany kontekst do zwartego formatu. Agent wywołuje to narzędzie, aby zapisać kluczowe preferencje, tak aby nawet jeśli starsze wiadomości zostaną usunięte, istotne informacje zostały zachowane.

Ten wzorzec jest podstawą do bardziej zaawansowanego redukowania historii:
1. Agent identyfikuje kluczowe fakty z rozmowy
2. Wywołuje narzędzie do podsumowywania, aby je zapisać
3. Starsze wiadomości można bezpiecznie usunąć, ponieważ podsumowanie zawiera to, co ważne

Poniżej definiujemy narzędzie `summarize_preferences`, które agent może wywołać, aby zapisać zwarte podsumowanie tego, czego się nauczył.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Podsumowanie

W tej lekcji nauczyłeś się, jak zarządzać kontekstem w długotrwałych rozmowach agentów przy użyciu Microsoft Agent Framework:

### Kluczowe Pojęcia
- **Okna kontekstowe są ograniczone** — każdy token w historii rozmowy generuje koszt i liczy się do limitu.
- **Narzędzia do podsumowywania** pozwalają agentowi skondensować zgromadzony kontekst w zwarte streszczenia, zmniejszając użycie tokenów przy zachowaniu istotnych informacji.
- **Notatniki agenta** zapewniają trwałą pamięć zewnętrzną, która przetrwa każde ograniczenie konwersacji.

### Czego się nauczyłeś
- **Agent świadomy kontekstu**, który utrzymuje ciągłość w rozmowach wielokrotnego kroku
- **Narzędzie do podsumowywania** (`summarize_preferences`), które zapisuje kluczowe informacje o użytkowniku w zwartej formie
- **Rozmowa wielokrotnego kroku**, demonstrująca zachowanie i zmianę kontekstu

### Zastosowania w rzeczywistości
- **Boty obsługi klienta**: Zapamiętują preferencje podczas długich sesji wsparcia
- **Osobiste asystenty**: Śledzą bieżące projekty bez konieczności powtarzania kontekstu
- **Nauczyciele edukacyjni**: Utrzymują postępy ucznia w wielu interakcjach

### Następne kroki
- Wdrożenie pełnoprawnego notatnika z trwałością opartą na plikach
- Dodanie automatycznego skracania historii po podsumowaniu
- Połączenie z bazami wektorowymi w celu semantycznego wyszukiwania w pamięci
- Tworzenie agentów, którzy mogą kontynuować rozmowy po dniach z pełnym kontekstem


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
